# 08 — Ablazione Strategia di Training: Balanced Random Pairs + Cosine Annealing LR (Senza Semi-Hard Mining)

## Motivazione Sperimentale
Nei Notebook `04`, `06`, `06b` e `07` abbiamo osservato un pattern ricorrente: nei 3 modelli migliori (`Opt 1 k=6`, `Opt 3b k=4,5,6`, `Opt 3b k=5,6,7`), il miglior checkpoint cadeva sempre a **`best_epoch = 0` (fine Warm-up)**, prima dell'attivazione del *Dynamic Semi-Hard Negative Mining*, che invece comprimeva il `distance_gap` da `~0.19` a `~0.15`.

In questo notebook testiamo **la stessa identica architettura `SiameseV3`** e **le stesse cache già salvate su Drive** (nessuna rigenerazione!) sostituendo il Semi-Hard Mining con:
- **12 epoche di `BalancedSiamesePairDataset`** (`32` anchor classe `0` + `32` anchor classe `1` per batch, 50% positive e 50% negative rimescolate ad ogni epoca);
- **`CosineAnnealingLR`** (da `5e-4` a `1e-5`), per una convergenza fluida e stabile del `distance_gap`.

Confrontiamo sullo stesso protocollo:
1. `siamese_cosine_baseline_k6` (Baseline $k=6$, `float32`, `937.5 MB`)
2. `siamese_cosine_opt1_rc_log_k6` (`Opt 1`: Bitwise + RC + Log, `uint8`, `234.4 MB`, `-75%` RAM)
3. `siamese_cosine_opt3b_enriched_ms_k456` (`Opt 3b`: Enriched O/E + Multi-Scala $k=4,5,6$, `uint8`, `703.1 MB`, `-25%` RAM, sparsità `<18%`)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import copy
import json
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'
VAL_POOL_PATH = PROCESSED_DIR / 'siamese_val_pair_pool.tsv'
TEST_POOL_PATH = PROCESSED_DIR / 'siamese_test_pair_pool.tsv'
FCGR_CACHE_DIR = PROCESSED_DIR / 'fcgr_cache'
FCGR_INDEX_PATH = FCGR_CACHE_DIR / 'fcgr_k6_index.tsv'

from src.dataset import (
    RandomSiamesePairDataset,
    BalancedSiamesePairDataset,
    SingleFCGRDataset,
)
from src.models import (
    SiameseV3,
    EuclideanContrastiveLoss,
    evaluate_pairwise,
    extract_embeddings,
    compute_topk_similarity_matrix,
)

print('PROJECT_ROOT :', PROJECT_ROOT)

In [ ]:
RANDOM_STATE = 42
EMBEDDING_DIM = 128
EUCLIDEAN_MARGIN = 1.25

EPOCHS = 12
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-4

BALANCED_BATCH_SIZE = 64
VAL_BATCH_SIZE = 128
VAL_PAIRS = 10_000
TEST_PAIRS = 10_000

LR_MAX = 5e-4
LR_MIN = 1e-5
WEIGHT_DECAY = 1e-4

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('Device :', DEVICE, f'({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else '')

metadata = pd.read_csv(MANIFEST_PATH, sep='\t', dtype={'id': str})
metadata['class_id'] = metadata['class_id'].astype(int)
train_metadata = metadata[metadata['split_cluster'] == 'train'].copy().reset_index(drop=True)
val_metadata = pd.read_csv(VAL_POOL_PATH, sep='\t', dtype={'id': str})
val_metadata['class_id'] = val_metadata['class_id'].astype(int)
test_metadata = pd.read_csv(TEST_POOL_PATH, sep='\t', dtype={'id': str})
test_metadata['class_id'] = test_metadata['class_id'].astype(int)

fcgr_index = pd.read_csv(FCGR_INDEX_PATH, sep='\t', dtype={'id': str})
id_to_fcgr_row = dict(zip(fcgr_index['id'], fcgr_index['fcgr_row']))

BATCHES_PER_EPOCH = math.ceil(len(train_metadata) / BALANCED_BATCH_SIZE)

def train_balanced_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    model.train()
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances = [], []
    t0 = time.perf_counter()
    for batch in loader:
        x1 = batch['x1'].to(DEVICE, non_blocking=True)
        x2 = batch['x2'].to(DEVICE, non_blocking=True)
        target = batch['target'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            z1, z2 = model(x1, x2)
        loss, distances = criterion(z1, z2, target)
        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        bs = target.shape[0]
        total_loss += loss.item() * bs
        total_samples += bs
        all_targets.append(target.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())
    targets = np.concatenate(all_targets)
    distances = np.concatenate(all_distances)
    return {'loss': total_loss / total_samples, 'auc': float(roc_auc_score(targets, -distances)), 'seconds': time.perf_counter() - t0}

def run_cosine_experiment(exp_name: str, fcgr_cache_path: Path, in_channels: int, k_order: int = 6):
    print('=' * 95)
    print(f'ESPERIMENTO COSINE (NO SEMI-HARD MINING): {exp_name} | k={k_order} | ch={in_channels}')
    print('=' * 95)
    out_dir = ARTIFACTS_DIR / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)
    fcgr_mm = np.load(fcgr_cache_path, mmap_mode='r')
    cache_mb = fcgr_cache_path.stat().st_size / (1024 ** 2)
    print(f'Cache: {fcgr_cache_path.name} | shape={fcgr_mm.shape} | dtype={fcgr_mm.dtype} | {cache_mb:.1f} MB')

    set_seed(RANDOM_STATE)
    train_ds = BalancedSiamesePairDataset(train_metadata, fcgr_mm, id_to_fcgr_row, BATCHES_PER_EPOCH, BALANCED_BATCH_SIZE, 0.50, RANDOM_STATE)
    val_ds = RandomSiamesePairDataset(val_metadata, fcgr_mm, id_to_fcgr_row, VAL_PAIRS, 0.50, RANDOM_STATE + 50_000)
    test_ds = RandomSiamesePairDataset(test_metadata, fcgr_mm, id_to_fcgr_row, TEST_PAIRS, 0.50, RANDOM_STATE + 60_000)

    train_loader = DataLoader(train_ds, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    test_loader = DataLoader(test_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)

    model = SiameseV3(embedding_dim=EMBEDDING_DIM, in_channels=in_channels).to(DEVICE)
    criterion = EuclideanContrastiveLoss(margin=EUCLIDEAN_MARGIN)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

    ckpt_path = out_dir / 'best_model.pt'
    best_auc = -1.0
    best_epoch = 0
    no_improve = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        curr_lr = optimizer.param_groups[0]['lr']
        tr = train_balanced_epoch(model, train_loader, train_ds, criterion, optimizer, scaler, epoch)
        va = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
        scheduler.step()
        history.append({'epoch': epoch, 'lr': curr_lr, 'train_auc': tr['auc'], **va})

        if va['auc'] > best_auc + MIN_DELTA:
            best_auc = va['auc']
            best_epoch = epoch
            no_improve = 0
            torch.save({'epoch': epoch, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': va}, ckpt_path)
            marker = ' <-- BEST'
        else:
            no_improve += 1
            marker = ''

        print(f"Epoch {epoch:02d}/{EPOCHS} (lr={curr_lr:.1e}) | train AUC {tr['auc']:.4f} | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s{marker}")
        if no_improve >= EARLY_STOPPING_PATIENCE:
            print('Early stopping attivato.')
            break

    pd.DataFrame(history).to_csv(out_dir / 'training_history.tsv', sep='\t', index=False)
    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt['model_state_dict'])
    model.eval()

    @torch.no_grad()
    def collect_dists(loader):
        t_all, d_all = [], []
        for b in loader:
            x1 = b['x1'].to(DEVICE, non_blocking=True)
            x2 = b['x2'].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
                z1, z2 = model(x1, x2)
            d = torch.linalg.vector_norm(z1.float() - z2.float(), ord=2, dim=1)
            t_all.append(b['target'].numpy())
            d_all.append(d.cpu().numpy())
        return np.concatenate(t_all), np.concatenate(d_all)

    val_t, val_d = collect_dists(val_loader)
    test_t, test_d = collect_dists(test_loader)
    thrs = np.linspace(val_d.min(), val_d.max(), 400)
    best_thr = float(thrs[int(np.argmax([f1_score(val_t, (val_d <= t).astype(int)) for t in thrs]))])
    test_preds_p = (test_d <= best_thr).astype(int)
    test_scores_p = -test_d
    pos_d, neg_d = test_d[test_t == 1], test_d[test_t == 0]
    gap_d = float(neg_d.mean() - pos_d.mean())
    d_prime_test = float(gap_d / np.sqrt(0.5 * (pos_d.var() + neg_d.var()) + 1e-12))

    pairwise_metrics = {
        'pairwise_roc_auc': float(roc_auc_score(test_t, test_scores_p)),
        'pairwise_pr_auc': float(average_precision_score(test_t, test_scores_p)),
        'pairwise_f1': float(f1_score(test_t, test_preds_p)),
        'pairwise_macro_f1': float(f1_score(test_t, test_preds_p, average='macro')),
        'pairwise_precision': float(precision_score(test_t, test_preds_p)),
        'pairwise_recall': float(recall_score(test_t, test_preds_p)),
        'pairwise_accuracy': float(accuracy_score(test_t, test_preds_p)),
        'd_pos_mean': float(pos_d.mean()),
        'd_neg_mean': float(neg_d.mean()),
        'distance_gap': gap_d,
        'd_prime': d_prime_test,
        'threshold': best_thr,
    }

    train_emb_loader = DataLoader(SingleFCGRDataset(train_metadata, fcgr_mm, id_to_fcgr_row), batch_size=256, shuffle=False)
    test_emb_loader = DataLoader(SingleFCGRDataset(test_metadata, fcgr_mm, id_to_fcgr_row), batch_size=256, shuffle=False)
    train_emb, train_lbl = extract_embeddings(model, train_emb_loader, DEVICE, AMP_ENABLED)
    test_emb, test_lbl = extract_embeddings(model, test_emb_loader, DEVICE, AMP_ENABLED)

    ref_per_class = min(5000, int((train_lbl == 0).sum()), int((train_lbl == 1).sum()))
    rng = np.random.default_rng(RANDOM_STATE)
    ref_idx = np.concatenate([
        rng.choice(np.where(train_lbl == 0)[0], size=ref_per_class, replace=False),
        rng.choice(np.where(train_lbl == 1)[0], size=ref_per_class, replace=False),
    ])
    sim_mat = compute_topk_similarity_matrix(test_emb, train_emb[ref_idx], train_lbl[ref_idx], n_classes=2, top_k=20)
    hotspot_score = sim_mat[:, 1] - sim_mat[:, 0]
    test_pred_cls = np.argmax(sim_mat, axis=1)

    cls_metrics = {
        'cls_roc_auc': float(roc_auc_score(test_lbl, hotspot_score)),
        'cls_pr_auc': float(average_precision_score(test_lbl, hotspot_score)),
        'cls_f1': float(f1_score(test_lbl, test_pred_cls)),
        'cls_macro_f1': float(f1_score(test_lbl, test_pred_cls, average='macro')),
        'cls_precision': float(precision_score(test_lbl, test_pred_cls)),
        'cls_recall': float(recall_score(test_lbl, test_pred_cls)),
        'cls_balanced_accuracy': float(balanced_accuracy_score(test_lbl, test_pred_cls)),
    }

    summary = {
        **pairwise_metrics,
        **cls_metrics,
        'k': k_order,
        'in_channels': in_channels,
        'cache_mb': round(cache_mb, 2),
        'train_samples': len(train_metadata),
        'best_epoch': best_epoch,
    }
    pd.DataFrame([summary]).to_csv(out_dir / f'{exp_name}_final_metrics.csv', index=False)
    with open(out_dir / f'{exp_name}_final_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=4)

    print(f'\n=== RISULTATI TEST SET: {exp_name} ===')
    pd.set_option('display.max_columns', None)
    display(pd.DataFrame([summary]))
    print(classification_report(test_lbl, test_pred_cls, target_names=['Stable (0)', 'Hotspot (1)'], digits=4))
    return summary

## 1. Run Cosine LR (No Semi-Hard): Baseline $k=6$ (`937.5 MB`)

In [ ]:
sum_base_cos = run_cosine_experiment(
    exp_name='siamese_cosine_baseline_k6',
    fcgr_cache_path=FCGR_CACHE_DIR / 'fcgr_k6.npy',
    in_channels=1,
    k_order=6,
)

## 2. Run Cosine LR (No Semi-Hard): `Opt 1` (`RC + Log`, `234.4 MB`, `-75%` RAM)

In [ ]:
sum_opt1_cos = run_cosine_experiment(
    exp_name='siamese_cosine_opt1_rc_log_k6',
    fcgr_cache_path=FCGR_CACHE_DIR / 'fcgr_opt_rc_log_k6_u8.npy',
    in_channels=1,
    k_order=6,
)

## 3. Run Cosine LR (No Semi-Hard): `Opt 3b` (`Enriched O/E + Multi-Scala k=4,5,6`, `703.1 MB`, `-25%` RAM)

In [ ]:
sum_opt3b_cos = run_cosine_experiment(
    exp_name='siamese_cosine_opt3b_enriched_ms_k456',
    fcgr_cache_path=FCGR_CACHE_DIR / 'fcgr_opt3b_enriched_ms_k456_u8.npy',
    in_channels=3,
    k_order=6,
)

## 4. Run Cosine LR (No Semi-Hard): `Opt 2` (`Multi-Scala k=4,5,6`, `703.1 MB`, `-25%` RAM)

In [ ]:
sum_opt2_cos = run_cosine_experiment(
    exp_name='siamese_cosine_opt2_multiscale_k456',
    fcgr_cache_path=FCGR_CACHE_DIR / 'fcgr_opt_multiscale_k456_u8.npy',
    in_channels=3,
    k_order=6,
)

## 5. Tabella Comparativa Completa (Semi-Hard Mining vs Cosine Balanced Random Pairs)

In [ ]:
pd.set_option('display.max_columns', None)
metric_files = sorted(ARTIFACTS_DIR.glob('siamese_*/*_final_metrics.csv'))
dfs = []
for f in metric_files:
    df_m = pd.read_csv(f)
    df_m.insert(0, 'experiment', f.parent.name)
    if f.parent.name == 'siamese_baseline_k6':
        df_m['in_channels'] = 1
        df_m['cache_mb'] = 937.50
    dfs.append(df_m)
full_df = pd.concat(dfs, ignore_index=True).sort_values('cls_roc_auc', ascending=False).reset_index(drop=True)
full_df.to_csv(COMPARISON_DIR / 'siamese_binary_comparison_table.csv', index=False)
cols_focus = [
    'experiment', 'k', 'in_channels', 'cache_mb', 'cls_roc_auc', 'cls_pr_auc', 'cls_f1', 'cls_balanced_accuracy',
    'pairwise_roc_auc', 'd_pos_mean', 'd_neg_mean', 'distance_gap', 'd_prime', 'best_epoch'
]
print('=== CLASSIFICA COMPLETA DI TUTTI GLI ESPERIMENTI (ORDINATI PER cls_roc_auc) ===')
display(full_df[[c for c in cols_focus if c in full_df.columns]])